# Hebrew Law Corpus → LLooM v2.1: semantic diagnostics + condition-faithful labels

This is the **v2.1** experiment runner. It reuses the passed-laws corpus and the
same shared preparation pipeline as the baseline (`lawsofisrael.extraction` +
`lawsofisrael.chunking`), but restructures the concept-discovery flow so that
**semantic diagnostics come before any density clustering**:

1. **Custom, context-rich summaries.** LLooM's generic default summary is
   replaced by a versioned free-form Hebrew prompt (`prompts_v2`) that keeps the
   details distinguishing one operative rule from another (affected population,
   actor, legal action, object, domain, exception/condition/timeframe) instead
   of collapsing to generic verbs like *תיקון חוק* or *הארכת תוקף*. The default
   quote filter is unchanged.
2. **Pre-clustering bullet artifact.** The exact bullets entering the embedding
   stage are captured with stable ids, chunk linkage, and honest accounting of
   how many chunks produced no bullet.
3. **Nearest-neighbor explorer.** The bullets are embedded with the local
   bge-m3 model and their cosine nearest neighbors are inspected **before** any
   UMAP/HDBSCAN, so we can judge whether legally similar provisions are close in
   the raw embedding space.
4. **Reproducible clustering diagnostics.** A seeded UMAP+HDBSCAN wrapper
   (`diagnostics_v2.ClusterConfig`) retains embeddings, coordinates, the fitted
   estimator, labels, membership probabilities, and outlier scores, plus the
   condensed-tree table/plot — **no cluster-size cutoff is baked in**.
5. **Deliberate selection → synthesis.** Only after inspecting diagnostics do we
   *manually* choose one clustering configuration; labels are synthesized from
   that result with the condition-faithful synthesis prompt.

**No scoring / no apply.** There is no `session.score`, no `scores_df`, no
`scores.parquet`. **Caveat:** a cluster label is a *shared legal pattern*, not
shared legislative intent (see the embedded caveat).

## 0. Setup

In [2]:
# Auto-reload edited package modules so changes to `lawsofisrael` are picked
# up without restarting the kernel (avoids stale-import AttributeErrors).
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import sys

import pandas as pd

In [3]:
# NOTE: if you started this kernel before updating the lawsofisrael package,
# restart the kernel once (Kernel -> Restart) so the latest module is loaded.

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from config import (
    MODEL_CONFIG, EMBEDDING_URL, EMBEDDING_MODEL, EMBEDDING_API_KEY,
    MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS,
    GEMINI_EMBEDDING_MODEL, GEMINI_EMBEDDING_DIM,
    GEMINI_EMBEDDING_BASE_URL, GEMINI_EMBEDDING_API_KEY,
)
from lawsofisrael import extraction, chunking, prompts_v2, v2_export
from lawsofisrael import diagnostics_v2 as diag
from lawsofisrael.lloom import make_lloom_session, build_gemini_embed_model

CACHE = ROOT / 'notebooks' / 'cache'
EXTRACTION_CACHE = CACHE / 'extraction'
OUTPUTS = ROOT / 'notebooks' / 'outputs'
DIAG_DIR = OUTPUTS / 'diagnostics'

CHAT_MODEL = MODEL_CONFIG['name']
CHAT_API_KEY = MODEL_CONFIG['api_key']
CHAT_BASE_URL = MODEL_CONFIG['base_url']
EMBED_URL = EMBEDDING_URL
EMBED_MODEL = EMBEDDING_MODEL
EMBED_API_KEY = EMBEDDING_API_KEY

CHUNK_TOKENS = chunking.compute_chunk_budget(MAX_CONTEXT_TOKENS, MAX_OUTPUT_TOKENS)

CORPUS_ID = 'nickbes/lawsofisrael'

# Corpus size. Density clustering needs enough bullets from enough distinct
# legal domains to have real similarity contrast; a handful of bills produces
# only same-bill neighbors and no usable clusters. The dataset has ~600 bills.
# Use a substantial subset here (fast enough to iterate, large enough for the
# diagnostics to be meaningful); set N_BILLS = None to run over ALL ~600 bills.
N_BILLS = 100

# Generation parameters (kept comparable to the v1 baseline).
GEN_PARAMS = {'filter_n_quotes': 5, 'summ_n_bullets': 7, 'synth_n_concepts': 3}
MAX_CONCEPTS = 30

# Nearest-neighbor explorer: how many neighbors to show per bullet.
NN_K = 5

# --- Embedding representation controls (diagnostics only) -------------------
# EMBED_BACKEND: 'gemini' = the API gemini-embedding model via the same
# OpenAI-compatible provider as the chat model (SENDS bullet text to the API);
# 'local' = local bge-m3 (session.cluster_model, no data leaves the machine).
# Default is 'gemini' for the current representation experiment; switch to
# 'local' if you must keep bullet text off the third-party API.
EMBED_BACKEND = 'gemini'
# Option 1: task instruction prefix. None = embed the bare bullet.
EMBED_INSTRUCTION = diag.DEFAULT_EMBED_INSTRUCTION
# Option 2: enrich each bullet with its chunk's domain (bill title + heading).
USE_DOMAIN_CONTEXT = True

## 1. Verify servers

In [4]:
import requests
from openai import OpenAI

if not CHAT_API_KEY:
    print("✗ OPENAI_API_KEY is not set. Copy .env.example to .env and fill it in.")

# The local bge-m3 server is only needed when the diagnostics embed locally.
# (LLooM's built-in clustering is NOT used in the v2.1 split flow, so the
# session's local cluster_model is never exercised for clustering.)
if EMBED_BACKEND == 'local':
    try:
        requests.get(f"{EMBED_URL}/health", timeout=5).raise_for_status()
        print("✓ Local embedding server running")
    except Exception as e:
        print(f"✗ Local embedding server not responding: {e}")
        print("  Start it with: ./scripts/start_embedding_server.sh")
else:
    print(f"ℹ EMBED_BACKEND={EMBED_BACKEND!r}: skipping local embedding-server check.")
    # Sanity-check the API embedding endpoint instead.
    try:
        _probe = build_gemini_embed_model(
            base_url=GEMINI_EMBEDDING_BASE_URL, api_key=GEMINI_EMBEDDING_API_KEY,
            model_name=GEMINI_EMBEDDING_MODEL, dimensions=GEMINI_EMBEDDING_DIM)
        _v, _ = _probe.fn(_probe, "בדיקה")
        print(f"✓ API embeddings ({GEMINI_EMBEDDING_MODEL}): dim={len(_v[0])}")
    except Exception as e:
        print(f"✗ API embedding model not responding: {e}")

chat = OpenAI(base_url=CHAT_BASE_URL, api_key=CHAT_API_KEY)
resp = chat.chat.completions.create(model=CHAT_MODEL, messages=[{"role": "user", "content": "השב בעברית: שלום"}], max_tokens=200)
print(f"✓ Chat API ({CHAT_MODEL}): {resp.choices[0].message.content}")

ℹ EMBED_BACKEND='gemini': skipping local embedding-server check.
✓ API embeddings (gemini-embedding-2-preview): dim=3072
✓ Chat API (gemini-3.5-flash-lite): שלום! מה שלומך? איך אפשר לעזור לך היום?


## 2. Load dataset and prepare (shared pipeline)

Extraction and chunking come from the shared `lawsofisrael` package — identical
to the baseline. Each chunk keeps its `source_spans` provenance for evidence
linking.

In [5]:
from datasets import load_dataset

dataset = load_dataset(CORPUS_ID, split='train')
df = dataset.to_pandas()
if N_BILLS:
    df = df.head(N_BILLS)
print(f"Preparing {len(df)} bills (N_BILLS={N_BILLS})")

records = [extraction.extract_bill(row.to_dict(), EXTRACTION_CACHE)
           for _, row in df.iterrows()]
success = sum(r['status'] == 'success' for r in records)
print(f"Extraction: {success}/{len(records)} successful")

chunk_objs = chunking.chunk_records(records, CHUNK_TOKENS)
chunk_records = chunking.chunks_to_records(chunk_objs)
chunks_df = pd.DataFrame(chunk_records)
assert chunks_df['tokens'].max() <= CHUNK_TOKENS, 'chunk exceeds token budget'
print(f"Built {len(chunks_df)} chunks from {chunks_df['bill_id'].nunique()} bills")
chunks_df[['chunk_id', 'bill_id', 'name', 'heading_path', 'page_no', 'tokens']].head()

Preparing 100 bills (N_BILLS=100)
Extraction: 100/100 successful
Built 100 chunks from 100 bills


,chunk_id,bill_id,name,heading_path,page_no,tokens
0,1057303:0000,1057303,חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) ...,ספר החוקים,1,2276
1,2229019:0000,2229019,חוק נוכחות עורך דין בחקירת קטינים ואנשים עם מו...,ספר החוקים,1,20418
2,2240475:0000,2240475,"חוק לתיקון פקודת העיריות (מס' 163), התשפ""ו–2026",ספר החוקים,1,1089
3,2240465:0000,2240465,"חוק התכנון והבנייה (תיקון מס' 170), התשפ""ו-2026",,1,724
4,1057405:0000,1057405,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום (ח...,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום )ח...,2,1212


## 3. Build LLooM session and v2.1 custom prompts

The chat + local embedding models are built by the shared `make_lloom_session`.
`prompts_v2.make_custom_prompts` supplies the default quote filter, the v2.1
**free-form context-rich summarize** prompt, and the condition-faithful
**synthesize** prompt. We record both prompt versions/hashes for the manifest.

In [6]:
import builtins, contextlib

@contextlib.contextmanager
def auto_confirm():
    orig = builtins.input
    builtins.input = lambda _: 'y'
    try:
        yield
    finally:
        builtins.input = orig

session = make_lloom_session(
    chunks_df,
    model_config=MODEL_CONFIG,
    chat_base_url=CHAT_BASE_URL,
    chat_api_key=CHAT_API_KEY,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    embed_url=EMBED_URL,
    embed_model_name=EMBED_MODEL,
    embed_api_key=EMBED_API_KEY,
)

custom_prompts = prompts_v2.make_custom_prompts(session)
PROMPT_INFO = prompts_v2.prompt_info()
SUMMARIZE_PROMPT_HASH = PROMPT_INFO['summarize_prompt_hash']
print("Summarize prompt:", PROMPT_INFO['summarize_prompt_version'], SUMMARIZE_PROMPT_HASH)
print("Synthesize prompt:", PROMPT_INFO['synthesize_prompt_version'], PROMPT_INFO['synthesize_prompt_hash'])

Summarize prompt: v2.1.2 e5d090a14eda
Synthesize prompt: v2.0.0 c141b3edbbe5


## 4. Distill only: quote filter + context-rich summaries

`diag.run_distill` runs **only** the distill stage (default quote filter, then
the v2.1 summary prompt). It stops before clustering so we can inspect the exact
bullets that will be embedded. This is the deliberate break in LLooM's usual
one-shot `gen`.

In [7]:
print(f"Distilling {len(chunks_df)} chunks (filter + context-rich summaries)...")
with auto_confirm():
    df_bullets = await diag.run_distill(session, custom_prompts, GEN_PARAMS)

print(f"Produced {len(df_bullets)} bullet rows")
df_bullets.head(10)

Distilling 100 chunks (filter + context-rich summaries)...
ERROR json_load on: ```json
{
    "relevant_quotes": [
        "חוק לתיקון פקודת העיריות )מס' 163(, התשפ"ו20261052 תיקון עקיף: חוק התכנון והבנייה, התשכ"ה19651965- מס' 171",
        "1 תיקון סעיף 252א . .1 בפקודת העיריות , בסעיף 252א, אחרי \"לא תטיל עירייה\" יבוא \"היטלים כמפורט להלן\"",
        "\")2( בתקופה שמיום ט"ז בסיוון התשפ"ו )1 ביוני 2026( עד יום ה' בטבת התשצ"א )31 בדצמבר 2030(, היטל בשל מערכת תיעול או בשל סלילת כבישים, מדרכות או רחובות, ובלבד שהמיתקן משמש לקירוי שטח או מבנה ושהיטלים כאמור שהוטלו בשל השטח או המבנה האמור - שולמו . \"",
        "2 1965 . .2 בתקופה של שנה מיום תחילתו של חוק זה, יקראו את חוק התכנון והבנייה, התשכ"ה-, כך שבתוספת השלישית, בסעיף 19)ב(, אחרי פסקה )14( יבוא:",
        "״)15( השבחה במקרקעין בשל הקמת מיתקן אגרו־וולטאי. . ״"
    ]
}
```
ERROR json_load on: {
    "relevant_quotes": [
        "חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום )חרבות ברזל( )פגישה עם עורך דין של עצור בעבירת ביטחון( )תיקון מס' 

,chunk_id,text
0,1057303:0000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...
1,1057303:0000,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...
2,1057303:0000,תיקון מספר 66 לפקודת בתי הסוהר במסגרת הוראת שע...
3,1057303:0000,הארכת תוקף הכרזה שניתנה לפי סעיף 19כ לפקודת בת...
4,2229019:0000,תיקון חוק הליכי חקירה והעדה מס' 5 לאנשים עם מו...
5,2229019:0000,תיקון חוק הנוער ענישה ודרכי טיפול הוראת שעה מס...
6,2229019:0000,התרת זכות לנוכחות עורך דין במהלך חקירה משטרתית
7,2229019:0000,התרת התחלת חקירה ללא המתנה להגעת עורך דין
8,2229019:0000,הסדרת התנהלותו של עורך דין הנוכח בפועל בחקירה
9,2240465:0000,הוספת המילה ממסחר אחרי מתעסוקה ברישה של פסקה 1


## 5. Pre-clustering bullet artifact + accounting

Capture the exact bullet corpus with stable ids and chunk linkage, and account
honestly for chunks that produced **no** bullet (a proxy for malformed/empty
model JSON that LLooM otherwise drops silently).

In [8]:
RUN_META = {'corpus_id': CORPUS_ID, 'summarize_prompt_hash': SUMMARIZE_PROMPT_HASH}
bullet_rows = diag.build_bullet_artifact(
    session, prompt_info=PROMPT_INFO, run_meta={'corpus_id': CORPUS_ID})
diag.validate_bullet_artifact(bullet_rows, chunk_ids=set(chunks_df['chunk_id']))

ACCOUNTING = diag.bullet_accounting(session)
print("Bullet accounting:", ACCOUNTING)

bullets_df = pd.DataFrame(bullet_rows)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
bullets_df.to_parquet(DIAG_DIR / 'bullets_v2.parquet', index=False)
print(f"Wrote {len(bullets_df)} bullets -> {DIAG_DIR / 'bullets_v2.parquet'}")
bullets_df[['bullet_row_id', 'chunk_id', 'bullet']].head(10)

Bullet accounting: {'n_input_chunks': 33, 'n_chunks_with_bullets': 33, 'n_chunks_without_bullets': 0, 'n_bullets': 170, 'chunk_ids_without_bullets': []}
Wrote 170 bullets -> /home/nick/Documents/projects/lawsofisrael/notebooks/outputs/diagnostics/bullets_v2.parquet


,bullet_row_id,chunk_id,bullet
0,1057303:0000#b000,1057303:0000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...
1,1057303:0000#b001,1057303:0000,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...
2,1057303:0000#b002,1057303:0000,תיקון מספר 66 לפקודת בתי הסוהר במסגרת הוראת שע...
3,1057303:0000#b003,1057303:0000,הארכת תוקף הכרזה שניתנה לפי סעיף 19כ לפקודת בת...
4,2229019:0000#b000,2229019:0000,תיקון חוק הליכי חקירה והעדה מס' 5 לאנשים עם מו...
5,2229019:0000#b001,2229019:0000,תיקון חוק הנוער ענישה ודרכי טיפול הוראת שעה מס...
6,2229019:0000#b002,2229019:0000,התרת זכות לנוכחות עורך דין במהלך חקירה משטרתית
7,2229019:0000#b003,2229019:0000,התרת התחלת חקירה ללא המתנה להגעת עורך דין
8,2229019:0000#b004,2229019:0000,הסדרת התנהלותו של עורך דין הנוכח בפועל בחקירה
9,2240465:0000#b000,2240465:0000,הוספת המילה ממסחר אחרי מתעסוקה ברישה של פסקה 1


In [9]:
# Trace a bullet back to its source chunk BEFORE any embedding/clustering.
chunk_lookup = v2_export.make_chunk_lookup(chunk_records)
if bullet_rows:
    b = bullet_rows[0]
    src = chunk_lookup.get(b['chunk_id'], {})
    print("BULLET:  ", b['bullet'])
    print("CHUNK_ID:", b['chunk_id'])
    print("BILL:    ", src.get('bill_id'), '-', src.get('name'))
    print("HEADING: ", src.get('heading_path'), '| page', src.get('page_no'))
    print("SOURCE (excerpt):", (src.get('text') or '')[:400])

BULLET:   הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם לחוק התשפ"ו-2026
CHUNK_ID: 1057303:0000
BILL:     1057303 - חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ"ו-2026
HEADING:  ספר החוקים | page 1
SOURCE (excerpt): ט"ו באב התשפ"ו 3575 29 ביולי 2026

עמוד

חוק לתיקון פקודת בתי הסוהר )הארכת הוראות שעה( )תיקוני חקיקה(, התשפ"ו20261048 תיקונים עקיפים: חוק לתיקון פקודת בתי הסוהר )מס' 64 - הוראת שעה - חרבות ברזל( )מצב חירום כליאתי(, התשפ"ד2023- - מס' 5 חוק לתיקון פקודת בתי הסוהר )תיקון מס' 66 - הוראת שעה - חרבות ברזל( )חופשה מיוחדת לאסיר(, התשפ"ד2024- - מס' 5

, בסעיף 1 -. .1 בחוק לתיקון פקודת בתי הסוהר )מס' 64 - ה


## 6. Nearest-neighbor explorer (summary bullets only)

Embed the bullets and inspect cosine nearest neighbors over the **raw** vectors
(self excluded) — before UMAP/HDBSCAN can distort the space. Three
representation controls (set in section 0) let you A/B what drives similarity:

- **`EMBED_BACKEND`** — `'local'` bge-m3 vs `'gemini'` API embeddings.
- **`EMBED_INSTRUCTION`** (Option 1) — a task prefix telling the embedder to
  represent the *legal effect / domain* rather than surface phrasing. Set to
  `None` to embed the bare bullet.
- **`USE_DOMAIN_CONTEXT`** (Option 2) — prepend each bullet's bill title +
  heading path so domain signal is injected explicitly.

Look at representative *extension*, *right*, and *prohibition* bullets: do their
neighbors now share population/domain/condition context, or merely a generic
verb / shared year token? Unrelated neighbors in a narrow similarity band mean
the representation still isn't separating legal concepts.

In [10]:
# Pick the embedding backend.
if EMBED_BACKEND == 'gemini':
    embed_model = build_gemini_embed_model(
        base_url=GEMINI_EMBEDDING_BASE_URL,
        api_key=GEMINI_EMBEDDING_API_KEY,
        model_name=GEMINI_EMBEDDING_MODEL,
        dimensions=GEMINI_EMBEDDING_DIM,
    )
    embed_backend_id = f"gemini:{GEMINI_EMBEDDING_MODEL}:dim={GEMINI_EMBEDDING_DIM or 'default'}"
    print(f"Embedding backend: {embed_backend_id} (bullet text is sent to the API)")
else:
    embed_model = session.cluster_model  # local bge-m3
    embed_backend_id = f"local:{EMBED_MODEL}"
    print(f"Embedding backend: {embed_backend_id} (local, no data leaves the machine)")

# Option 2: per-chunk domain context (bill title + heading path).
context_by_id = chunk_lookup if USE_DOMAIN_CONTEXT else None
# make_chunk_lookup rows use 'name' for the bill title; embed_bullets reads
# 'bill_title' or 'name', so the raw chunk records work directly.

bullet_ids, bullet_texts, bullet_emb = diag.embed_bullets(
    bullet_rows, embed_model,
    instruction=EMBED_INSTRUCTION, context_by_id=context_by_id)
neighbor_rows = diag.nearest_neighbors(bullet_ids, bullet_texts, bullet_emb, k=NN_K)
neighbors_df = pd.DataFrame(neighbor_rows)
neighbors_df.to_parquet(DIAG_DIR / 'nearest_neighbors_v2.parquet', index=False)
print(f"Wrote nearest neighbors (k={NN_K}) -> {DIAG_DIR / 'nearest_neighbors_v2.parquet'}")

def show_neighbors(bullet_id):
    for r in diag.neighbors_for(neighbor_rows, bullet_id):
        print(f"  [{r['rank']}] sim={r['similarity']:.3f}  {r['neighbor_bullet']}")

# Inspect a few probe bullets by keyword (extension / right / prohibition).
import re
def find_bullet(keyword):
    for bid, txt in zip(bullet_ids, bullet_texts):
        if re.search(keyword, txt):
            return bid, txt
    return None, None

for kw in ['הארכ', 'זכות', 'איסור']:  # extension / right / prohibition
    bid, txt = find_bullet(kw)
    if bid:
        print(f"\nQUERY ({kw}): {txt}")
        show_neighbors(bid)
neighbors_df.head(10)

Embedding backend: gemini:gemini-embedding-2-preview:dim=3072 (bullet text is sent to the API)
Wrote nearest neighbors (k=5) -> /home/nick/Documents/projects/lawsofisrael/notebooks/outputs/diagnostics/nearest_neighbors_v2.parquet

QUERY (הארכ): הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם לחוק התשפ"ו-2026
  [1] sim=0.967  הארכת תוקף הכרזה שניתנה לפי סעיף 19כ לפקודת בתי הסוהר עד ליום ט"ו בספטמבר 2026
  [2] sim=0.961  תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שעה למצב חירום כליאתי חרבות ברזל
  [3] sim=0.946  תיקון מספר 66 לפקודת בתי הסוהר במסגרת הוראת שעה לחופשה מיוחדת לאסיר בחרבות ברזל
  [4] sim=0.906  הארכת תוקף תחולה והוראת שעה עד ליום 30 ביוני 2029 במקום 1 ביולי 2024
  [5] sim=0.901  תיקון חוק קיום דיונים בהיוועדות חזותית לעצורים אסירים וכלואים הוראת שעה

QUERY (זכות): התרת זכות לנוכחות עורך דין במהלך חקירה משטרתית
  [1] sim=0.930  התרת התחלת חקירה ללא המתנה להגעת עורך דין
  [2] sim=0.927  הסדרת התנהלותו של עורך דין הנוכח בפועל בחקירה
  [3] sim=0.893  תיקון חוק הליכי חקירה והעדה 

,bullet_row_id,bullet,rank,neighbor_id,neighbor_bullet,similarity
0,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,1,1057303:0000#b003,הארכת תוקף הכרזה שניתנה לפי סעיף 19כ לפקודת בת...,0.967302
1,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,2,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,0.961165
2,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,3,1057303:0000#b002,תיקון מספר 66 לפקודת בתי הסוהר במסגרת הוראת שע...,0.946062
3,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,4,2220910:0000#b001,הארכת תוקף תחולה והוראת שעה עד ליום 30 ביוני 2...,0.905710
4,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,5,1057405:0000#b001,תיקון חוק קיום דיונים בהיוועדות חזותית לעצורים...,0.901389
5,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,1,1057303:0000#b002,תיקון מספר 66 לפקודת בתי הסוהר במסגרת הוראת שע...,0.961595
6,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,2,1057303:0000#b000,הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתאם...,0.961165
7,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,3,1057303:0000#b003,הארכת תוקף הכרזה שניתנה לפי סעיף 19כ לפקודת בת...,0.944933
8,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,4,1057405:0000#b000,תיקון חוק לתיקון ולהארכת תוקפן של תקנות שעת חי...,0.907259
9,1057303:0000#b001,תיקון מספר 64 לפקודת בתי הסוהר במסגרת הוראת שע...,5,1057405:0000#b001,תיקון חוק קיום דיונים בהיוועדות חזותית לעצורים...,0.902378


## 7. Reproducible clustering diagnostics — compare configurations

Run one or more seeded UMAP+HDBSCAN configurations. **No cluster-size cutoff is
prescribed**: define a few `ClusterConfig`s and compare their noise ratio,
cluster count, membership probabilities, and condensed-tree stability. Every run
retains raw embeddings, UMAP coordinates, the fitted estimator, labels,
probabilities, and outlier scores, and writes its condensed-tree table (plus an
optional plot and UMAP scatter, if matplotlib is installed).

In [18]:
# Candidate configurations to compare. Edit freely — this is exploration.
configs = {
    'A': diag.ClusterConfig(hdbscan_min_cluster_size=3, hdbscan_cluster_selection_method='leaf', random_state=42),
    'B': diag.ClusterConfig(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method='eom', random_state=42),
    'C': diag.ClusterConfig(hdbscan_min_cluster_size=5, hdbscan_cluster_selection_method='leaf', random_state=42),
    'D': diag.ClusterConfig(hdbscan_min_cluster_size=7, hdbscan_cluster_selection_method='eom', random_state=42),
    'E': diag.ClusterConfig(hdbscan_min_cluster_size=4, hdbscan_cluster_selection_method='leaf', random_state=42)
}


results = {}
for name, cfg in configs.items():
    try:
        res = diag.run_clustering(bullet_ids, bullet_texts, bullet_emb, cfg,
                                  bullet_prompt_hash=SUMMARIZE_PROMPT_HASH)
    except ValueError as e:
        print(f"[{name}] skipped: {e}")
        continue
    results[name] = res
    entry = diag.clustering_manifest_entry(res)
    print(f"[{name}] run_id={entry['cluster_run_id']} "
          f"n_clusters={entry['n_clusters']} noise={entry['noise_ratio']:.0%} "
          f"cross_bill={entry['n_cross_bill_clusters']} "
          f"single_bill={entry['n_single_bill_clusters']}")
    rid = res.config.run_id()
    diag.condensed_tree_table(res).to_parquet(DIAG_DIR / f'condensed_tree_{name}_{rid}.parquet', index=False)
    tree_png = diag.save_condensed_tree_plot(res, str(DIAG_DIR / f'condensed_tree_{name}_{rid}.png'))
    scatter_png = diag.save_umap_scatter(res, str(DIAG_DIR / f'umap_scatter_{name}_{rid}.png'))
    if tree_png is None:
        print(f"[{name}] (matplotlib not installed — condensed-tree table written, plot skipped)")

# Per-config cluster composition: distinct bills per cluster is the artifact
# check. single_bill=True means a cluster is one bill's provisions, not a
# cross-bill legal concept.
for name, res in results.items():
    print(f"\n=== config {name} ({res.config.run_id()}) — cluster composition ===")
    comp = pd.DataFrame(res.cluster_composition())
    if len(comp):
        display(comp[['cluster_id', 'size', 'n_bills', 'single_bill', 'mean_prob']])
    else:
        print("  (no clusters — all noise)")

# Full per-bullet assignment (with bill_id) for inspection:
for name, res in results.items():
    print(f"\n=== config {name} — per-bullet (head) ===")
    display(pd.DataFrame(res.diagnostic_table())
            .sort_values(['cluster_id', 'bill_id']).head(20))


resB = results['E']
tbl = pd.DataFrame(resB.diagnostic_table())
for cid in [12, 3, 8, 6]:   # the 4-5 bill clusters
    print(f"\n=== cluster {cid} ===")
    for _, r in tbl[tbl.cluster_id == cid].sort_values('bill_id').iterrows():
        print(f"  [{r['bill_id']}] {r['bullet']}")



/home/nick/Documents/projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[A] run_id=umap15x5_mcs3_leaf_seed42 n_clusters=24 noise=8% cross_bill=6 single_bill=18
[A] (matplotlib not installed — condensed-tree table written, plot skipped)


/home/nick/Documents/projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[B] run_id=umap15x5_mcs5_eom_seed42 n_clusters=14 noise=15% cross_bill=8 single_bill=6
[B] (matplotlib not installed — condensed-tree table written, plot skipped)


/home/nick/Documents/projects/lawsofisrael/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...


=== config D — per-bullet (head) ===


,bullet_row_id,bill_id,bullet,cluster_id,membership_prob,outlier_score,umap_x,umap_y
154,1046680:0000#b000,1046680,תיקון חוק התכנון והבנייה במסגרת תיקון מס' 164,-1,0.0,0.586783,0.884926,3.081514
156,1046680:0000#b002,1046680,תיקון חוק הרשות הארצית לכבאות והצלה במס' 13,-1,0.0,0.608642,0.819279,3.059726
87,2197936:0000#b000,2197936,החלפת המילים עשרים ושמונה בשלושים ושלושים וארב...,-1,0.0,0.541120,1.040890,3.817379
88,2197936:0000#b001,2197936,החלפת המילים עשרים ושלוש בשלושים ושלוש בסעיף ק...,-1,0.0,0.571941,0.941608,3.752281
96,2197937:0000#b000,2197937,הקמת מחלקה לחקירת שוטרים במשרד המשפטים בידי שר...,-1,0.0,0.626324,0.085992,2.822776
97,2197937:0000#b001,2197937,קביעת כשירות מנהל המחלקה לבעל ניסיון פלילי ושו...,-1,0.0,0.635751,0.054991,2.848231
98,2197937:0000#b002,2197937,הגשת כתבי אישום בעבירות שנחקרו במחלקה בידי תוב...,-1,0.0,0.634417,0.030661,2.862739
99,2197937:0000#b003,2197937,הכרעה במחלוקת בין המחלקה לגוף חוקר בידי הממונה...,-1,0.0,0.620146,0.085570,2.814135
100,2197937:0000#b004,2197937,העברת תיקי חקירה ותביעה בין המחלקה לגופים אחרי...,-1,0.0,0.644849,-0.001490,2.892823
114,2220439:0000#b000,2220439,קידום מתן סיוע והסדרת זכויות בני משפחה של נעדרים,-1,0.0,0.674854,2.416214,2.526361



=== config E — per-bullet (head) ===


,bullet_row_id,bill_id,bullet,cluster_id,membership_prob,outlier_score,umap_x,umap_y
87,2197936:0000#b000,2197936,החלפת המילים עשרים ושמונה בשלושים ושלושים וארב...,-1,0.000000,0.822227,1.040890,3.817379
88,2197936:0000#b001,2197936,החלפת המילים עשרים ושלוש בשלושים ושלוש בסעיף ק...,-1,0.000000,0.821975,0.941608,3.752281
41,2202055:0000#b000,2202055,הוספת דודנם של בני המשפחה להגדרת בן משפחה בחוק...,-1,0.000000,0.704302,0.572808,3.300297
42,2202055:0000#b001,2202055,החלת תיקון סעיף 18א על תביעות שטרם התיישנו ערב...,-1,0.000000,0.758144,0.615558,3.199354
39,2220910:0000#b002,2220910,קביעת דמי קיום בגובה כפל דמי הקיום הקבועים לפי...,-1,0.000000,0.831490,1.610468,3.235729
40,2220910:0000#b003,2220910,קביעת דמי קיום בשיעור 175% מדמי הקיום הקבועים ...,-1,0.000000,0.837553,1.588100,3.248947
6,2229019:0000#b002,2229019,התרת זכות לנוכחות עורך דין במהלך חקירה משטרתית,-1,0.000000,0.653128,-0.175177,3.095230
7,2229019:0000#b003,2229019,התרת התחלת חקירה ללא המתנה להגעת עורך דין,-1,0.000000,0.641487,-0.281108,3.149827
8,2229019:0000#b004,2229019,הסדרת התנהלותו של עורך דין הנוכח בפועל בחקירה,-1,0.000000,0.625074,-0.104808,3.104239
107,2225412:0000#b000,2225412,קביעת שר הפנים שמסתנן תומך ממשל יוצרת חזקה להי...,0,1.000000,0.000000,2.445677,4.192976



=== cluster 12 ===
  [2199298] קביעת עונש מוות למחבלים שביצעו פיגועי טרור רצחניים כאמצעי מאבק
  [2199298] הטלת עונש מוות על תושב האזור הגורם בכוונה למות אדם במעשה טרור
  [2199298] ביצוע עונש מוות שגזר בית משפט צבאי בידי שירות בתי הסוהר
  [2199298] ביצוע גזר דין מוות בעבירות ביטחון בתוך 90 ימים מהיותו לחלוט
  [2199298] הטלת עונש מיתה או מאסר עולם על גרימת מוות בכוונה לשלוילת קיומה של מדינת ישראל
  [2201009] קביעת עונש מאסר חמש שנים בשל עבירה בחלק מהותי בנשק
  [2201009] קביעת עונש מאסר שלוש שנים בשל עבירה בחלק נשק שאינו מהותי
  [2201009] חובת בית המשפט להורות על חילוט רכוש לאוצר המדינה בהרשעה

=== cluster 3 ===
  [2216066] תיקון חוק הרשות השנייה לטלוויזיה ורדיו במסגרת תיקון מספר 50
  [2216066] הסדרת נושא תיקון סעיף 34 הנוגע לרישוי שידורים
  [2216066] קביעת הוראות ספציפיות לעניין רישוי שידורי רדיו אנלוגי
  [2216066] מתן חובה לפרסום מכרז ראשון למתן רישיון לשידורי רדיו אנלוגי
  [2216066] הכללת הוראות מעבר במסגרת התיקון לחוק

=== cluster 8 ===
  [1043724] הגדרת בעל מלאי כמי שהחזיק כמויות סף

## 8. Decision point — choose ONE configuration deliberately

**This is the selection mechanism.** Having inspected nearest neighbors and the
clustering diagnostics above, set `CHOSEN_CONFIG` to the key of the configuration
you judge appropriate — or leave it `None` to stay in an exploratory, *no-label*
state. Nothing is synthesized while `CHOSEN_CONFIG is None`.

In [19]:
CHOSEN_CONFIG = 'E'   # e.g. 'A' — set only after inspecting the diagnostics above.

if CHOSEN_CONFIG is None:
    chosen_result = None
    print("Exploratory state: no configuration chosen -> NO labels will be synthesized.")
    print("Inspect the diagnostics above, then set CHOSEN_CONFIG to a config key and re-run from here.")
else:
    chosen_result = results[CHOSEN_CONFIG]
    # Guard: the chosen result must match the bullets we actually inspected.
    diag.guard_selection(chosen_result, expected_bullet_prompt_hash=SUMMARIZE_PROMPT_HASH)
    print(f"Chosen config {CHOSEN_CONFIG}: {chosen_result.config.run_id()} "
          f"({chosen_result.n_clusters()} clusters, noise {chosen_result.noise_ratio():.0%})")

Chosen config E: umap15x5_mcs4_leaf_seed42 (21 clusters, noise 5%)


## 9. Synthesize labels from the chosen clustering (no scoring)

`diag.synthesize_from_result` re-guards against a mismatched bullet corpus,
builds the per-cluster input from the **chosen** result only, synthesizes with
the condition-faithful prompt, reviews, and selects. The pipeline ends here:
there is deliberately no `session.score`.

In [20]:
if chosen_result is None:
    print("No configuration chosen — skipping synthesis (exploratory state).")
    concepts_table = []
else:
    with auto_confirm():
        selected_ids = await diag.synthesize_from_result(
            session, chosen_result, custom_prompts, GEN_PARAMS,
            expected_bullet_prompt_hash=SUMMARIZE_PROMPT_HASH,
            max_concepts=MAX_CONCEPTS,
        )
    v2_export.assert_no_scoring(session)  # guard: v2.1 must not score
    concepts_table = v2_export.build_concepts_table(session)
    print(f"Selected {len(selected_ids)} labels")
pd.DataFrame(concepts_table)

Selected 30 labels


,concept_id,label,criterion,rep_chunk_ids
0,963593f4-0e3c-43db-9faf-497f22a7049b,"הארכת תוקף, תיקון או החלת הוראת שעה לגבי בתי ס...","האם הטקסט עוסק בהארכת תוקף, תיקון או קביעה של ...","[1057303:0000, 1057405:0000]"
1,a4cbfc70-ad0e-409a-8de2-90fee4909ce5,תיקון חקיקה או הארכת תוקף הוראת שעה בענייני קט...,האם הטקסט עוסק בתיקון חוק או בהוראת שעה הנוגעת...,[2229019:0000]
2,356272b8-d263-4fd1-ac82-73a628251be7,הארכת תוקף או תיקון הוראת שעה וסעיפים הנוגעים ...,"האם הטקסט עוסק בהארכת תוקף, תחולה או תיקון של ...","[1057200:0000, 2220910:0000]"
3,fe852204-3251-4052-a0a3-494a893ef6be,קביעת הוראות מיוחדות והיערכות מיוחדת לבחירות ב...,האם הטקסט קובע הוראות מיוחדות או מסמיך גורמים ...,[2244462:0000]
4,d3ed196a-64df-46a1-90b9-b16b72723a0d,הסדרת זכות הצבעה בקלפי ייעודית או נגישה לבעלי ...,האם הטקסט מקנה זכות הצבעה ומסדיר את האפשרות לה...,[2244462:0000]
5,c230e000-d4ef-465e-9312-afa54e65a1e2,חובת צירוף גילוי ברור ובולט על תעמולת בחירות ש...,האם הטקסט מטיל חובה לצרף גילוי ברור ובולט על ח...,[2244462:0000]
6,e7af7a31-a2bf-486b-a397-f9fd437a8c62,הקמת הרשות לתקשורת משודרת כגוף חדש וכאשכול משפ...,האם הטקסט קובע את הקמתה של הרשות לתקשורת משודר...,[1042100:0000]
7,eb03bd33-d12e-4df3-9016-8eaead86be3b,איסור עיסוק או פעילות ללא רישיון או היתר כנדרש,האם הטקסט קובע איסור מפורש על אדם או גוף לעסוק...,"[1046571:0000, 2204244:0000]"
8,7a5cfc75-55a9-47c2-b0b3-790416ec5e33,הסמכת רשויות או גורמים לבצע פעולות ואכיפה לפי חוק,האם הטקסט מעניק סמכות פעולה או אכיפה לגוף מוסמ...,"[1046571:0000, 1046237:0000]"
9,7d11e64c-e4a6-46b8-8978-5e8d2854c518,תוקף משפטי של ייפוי כוח לתאגיד או לעסקה ללא ער...,האם הטקסט קובע כי ייפוי כוח שניתן לתאגיד או לב...,[2232915:0000]


## 10. Run manifest (reproducibility)

In [21]:
clustering_info = None
artifact_paths = {
    'bullets': str((DIAG_DIR / 'bullets_v2.parquet').relative_to(ROOT)),
    'nearest_neighbors': str((DIAG_DIR / 'nearest_neighbors_v2.parquet').relative_to(ROOT)),
}
if chosen_result is not None:
    rid = chosen_result.config.run_id()
    clustering_info = diag.clustering_manifest_entry(chosen_result)
    clustering_info['chosen_config_key'] = CHOSEN_CONFIG
    artifact_paths['condensed_tree'] = f'notebooks/outputs/diagnostics/condensed_tree_{CHOSEN_CONFIG}_{rid}.parquet'

manifest = v2_export.build_run_manifest(
    corpus_id=CORPUS_ID,
    n_bills=N_BILLS,
    n_chunks=len(chunks_df),
    chat_model=CHAT_MODEL,
    embed_model=EMBED_MODEL,
    lloom_version=version('text_lloom'),
    gen_params=GEN_PARAMS,
    prompt_info=PROMPT_INFO,
    timestamp=datetime.now(timezone.utc).isoformat(),
    experiment='lloom_v2.1_semantic_diagnostics',
    bullet_accounting=ACCOUNTING,
    nearest_neighbor_info={
        'k': NN_K, 'metric': 'cosine',
        'embed_backend': embed_backend_id,
        'instruction': EMBED_INSTRUCTION,
        'domain_context': bool(USE_DOMAIN_CONTEXT),
    },
    clustering_info=clustering_info,
    artifact_paths=artifact_paths,
)
manifest

{'experiment': 'lloom_v2.1_semantic_diagnostics',
 'corpus_id': 'nickbes/lawsofisrael',
 'n_bills': 100,
 'n_chunks': 100,
 'chat_model': 'gemini-3.5-flash-lite',
 'embed_model': 'bge-m3',
 'lloom_version': '0.8.2.1',
 'gen_params': {'filter_n_quotes': 5,
  'summ_n_bullets': 7,
  'synth_n_concepts': 3},
 'scored': False,
 'cluster_label_caveat': "Cluster labels describe a shared legal PATTERN (operative effect and explicit scope/conditions) across the enacted-law text of multiple bills. They do NOT establish shared legislative INTENT: final enacted-law text can support statements about legal effect and explicit scope, but cannot independently establish the government's unexpressed purpose. Trace each label to its exported source spans before drawing conclusions.",
 'timestamp': '2026-08-29T10:57:25.893939+00:00',
 'summarize_prompt_version': 'v2.1.2',
 'summarize_prompt_hash': 'e5d090a14eda',
 'synthesize_prompt_version': 'v2.0.0',
 'synthesize_prompt_hash': 'c141b3edbbe5',
 'bullet_ac

## 11. Export labels + provenance (for manual exploration)

If a configuration was chosen, write the label + provenance artifacts to
`notebooks/outputs/`. In the exploratory (no-config) state, **no labels are
written** — only the diagnostic artifacts from the sections above. There is
never a `scores.parquet`.

In [22]:
OUTPUTS.mkdir(parents=True, exist_ok=True)
(OUTPUTS / 'run_manifest_v2.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

if not concepts_table:
    print("Exploratory state: no labels_v2.parquet / label_provenance_v2.parquet written.")
    print("Wrote diagnostics + run_manifest_v2.json only.")
    provenance_df = pd.DataFrame()
else:
    quotes_lookup, bullets_lookup = v2_export.distill_lookups(session)
    membership_table = v2_export.build_membership_table(
        session, chunk_lookup, quotes_lookup=quotes_lookup, bullets_lookup=bullets_lookup)
    v2_export.validate_provenance(concepts_table, membership_table)

    labels_df = pd.DataFrame(concepts_table)
    provenance_df = pd.DataFrame(membership_table)
    labels_df.to_parquet(OUTPUTS / 'labels_v2.parquet', index=False)
    provenance_df.to_parquet(OUTPUTS / 'label_provenance_v2.parquet', index=False)
    print(f"Exported labels_v2.parquet, label_provenance_v2.parquet, run_manifest_v2.json to {OUTPUTS}")

print(v2_export.CLUSTER_LABEL_CAVEAT)
provenance_df.head()

Exported labels_v2.parquet, label_provenance_v2.parquet, run_manifest_v2.json to /home/nick/Documents/projects/lawsofisrael/notebooks/outputs
Cluster labels describe a shared legal PATTERN (operative effect and explicit scope/conditions) across the enacted-law text of multiple bills. They do NOT establish shared legislative INTENT: final enacted-law text can support statements about legal effect and explicit scope, but cannot independently establish the government's unexpressed purpose. Trace each label to its exported source spans before drawing conclusions.


,concept_id,label,criterion,chunk_id,resolved,bill_id,bill_title,heading_path,page_no,source_spans,source_text,quotes,bullets
0,963593f4-0e3c-43db-9faf-497f22a7049b,"הארכת תוקף, תיקון או החלת הוראת שעה לגבי בתי ס...","האם הטקסט עוסק בהארכת תוקף, תיקון או קביעה של ...",1057303:0000,True,1057303,חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) ...,ספר החוקים,1,"[{'block_id': '1057303:b0001', 'heading_path':...","ט""ו באב התשפ""ו 3575 29 ביולי 2026\n\nעמוד\n\nח...",[חוק לתיקון פקודת בתי הסוהר )הארכת הוראות שעה(...,[הארכת הוראות השעה לתיקון פקודת בתי הסוהר בהתא...
1,963593f4-0e3c-43db-9faf-497f22a7049b,"הארכת תוקף, תיקון או החלת הוראת שעה לגבי בתי ס...","האם הטקסט עוסק בהארכת תוקף, תיקון או קביעה של ...",1057405:0000,True,1057405,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום (ח...,חוק לתיקון ולהארכת תוקפן של תקנות שעת חירום )ח...,2,"[{'block_id': '1057405:b0002', 'heading_path':...",. .1 בחוק לתיקון ולהארכת תוקפן של תקנות שעת חי...,[בחוק לתיקון ולהארכת תוקפן של תקנות שעת חירום ...,[תיקון חוק לתיקון ולהארכת תוקפן של תקנות שעת ח...
2,a4cbfc70-ad0e-409a-8de2-90fee4909ce5,תיקון חקיקה או הארכת תוקף הוראת שעה בענייני קט...,האם הטקסט עוסק בתיקון חוק או בהוראת שעה הנוגעת...,2229019:0000,True,2229019,חוק נוכחות עורך דין בחקירת קטינים ואנשים עם מו...,ספר החוקים,1,"[{'block_id': '2229019:b0001', 'heading_path':...","ט""ו באב התשפ""ו 3578 29 ביולי 2026\n\nעמוד\n\nס...",[תיקון חוק הליכי חקירה והעדה )התאמה לאנשים עם ...,[תיקון חוק הליכי חקירה והעדה מס' 5 לאנשים עם מ...
3,356272b8-d263-4fd1-ac82-73a628251be7,הארכת תוקף או תיקון הוראת שעה וסעיפים הנוגעים ...,"האם הטקסט עוסק בהארכת תוקף, תחולה או תיקון של ...",1057200:0000,True,1057200,חוק לתיקון פקודת סדר הדין הפלילי (מעצר וחיפוש)...,ספר החוקים,1,"[{'block_id': '1057200:b0001', 'heading_path':...","ו' באב התשפ""ו 3553 20 ביולי 2026\n\nעמוד\n\nחו...",[חוק לתיקון פקודת סדר הדין הפלילי )מעצר וחיפוש...,[התרת חיפוש ללא צו במקום או בבית בהתקיים חשד ס...
4,356272b8-d263-4fd1-ac82-73a628251be7,הארכת תוקף או תיקון הוראת שעה וסעיפים הנוגעים ...,"האם הטקסט עוסק בהארכת תוקף, תחולה או תיקון של ...",2220910:0000,True,2220910,"חוק שירות ביטחון (תיקון מס' 29 - הוראת שעה), ה...",,1,"[{'block_id': '2220910:b0000', 'heading_path':...","ספר החוקים\n\nה' באב התשפ""ו 3552 19 ביולי 2026...",[תיקון סעיף 15 -הוראת שעה\nיוצא צבא שהיה בשירו...,[תיקון סעיף 15 במסגרת הוראת שעה לגבי יוצא צבא ...


## 12. Trace a label: label → cluster → bullet neighbors → source chunk

For one selected label, walk the full evidence chain: the label, the chosen
clustering config it came from, the LLooM reasoning path (filter quotes + summary
bullets), the bullet's nearest neighbors, and the authoritative source text with
bill title/heading/page. This works only when a configuration was chosen.

In [23]:
if len(provenance_df):
    row = provenance_df.iloc[0]
    print("LABEL:    ", row['label'])
    print("CRITERION:", row['criterion'])
    print("CLUSTER:  ", chosen_result.config.run_id() if chosen_result else None)
    print("BILL:     ", row['bill_id'], '-', row['bill_title'])
    print("HEADING:  ", row['heading_path'], '| page', row['page_no'])

    print("\nFILTER QUOTES (LLooM-extracted):")
    for q in (row['quotes'] or []):
        print("  -", q)
    print("\nSUMMARY BULLETS (LLooM-generated):")
    for bl in (row['bullets'] or []):
        print("  -", bl)

    # Nearest neighbors of the chunk's first bullet, if present.
    first_bid = diag.bullet_row_id(row['chunk_id'], 0)
    nn = diag.neighbors_for(neighbor_rows, first_bid)
    if nn:
        print("\nNEAREST BULLET NEIGHBORS:")
        for r in nn:
            print(f"  [{r['rank']}] sim={r['similarity']:.3f}  {r['neighbor_bullet']}")

    print("\nSOURCE TEXT (authoritative, excerpt):")
    print((row['source_text'] or '')[:800])
else:
    print("No selected labels to trace (exploratory state or empty selection).")

LABEL:     הארכת תוקף, תיקון או החלת הוראת שעה לגבי בתי סוהר, עצורים ואסירים במצב חירום
CRITERION: האם הטקסט עוסק בהארכת תוקף, תיקון או קביעה של הוראת שעה הנוגעת לפקודת בתי הסוהר, לעצורים, לאסירים או לתנאי כליאה בשעת חירום?
CLUSTER:   umap15x5_mcs4_leaf_seed42
BILL:      1057303 - חוק לתיקון פקודת בתי הסוהר (הארכת הוראות שעה) (תיקוני חקיקה), התשפ"ו-2026
HEADING:   ספר החוקים | page 1

FILTER QUOTES (LLooM-extracted):
  - חוק לתיקון פקודת בתי הסוהר )הארכת הוראות שעה( )תיקוני חקיקה(, התשפ"ו2026
חוק לתיקון פקודת בתי הסוהר )מס' 64 - הוראת שעה - חרבות ברזל( )מצב חירום כליאתי(, התשפ"ד2023- - מס' 5
חוק לתיקון פקודת בתי הסוהר )תיקון מס' 66 - הוראת שעה - חרבות ברזל( )חופשה מיוחדת לאסיר(, התשפ"ד2024- - מס' 5
הוראת מעבר . .3 הכרזה שניתנה לפני תחילתו של חוק זה לפי סעיף 19כ לפקודה, כנוסחו בהוראת השעה האמורה בסעיף 1 לחוק זה, מוארכת ותמשיך לעמוד בתוקפה עד יום ד' בתשרי התשפ"ז )15 בספטמבר 2026( .
איתמר בן גביר השר לביטחון לאומי

SUMMARY BULLETS (LLooM-generated):
  - הארכת הוראות השעה לתיקון פקודת בתי 